In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')
import random
from tqdm import tqdm

def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Configuration
NUM_SEEDS = 5
SEEDS = [42, 123, 456, 789, 999]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Testing {NUM_SEEDS} different seeds")
print(f"Focus: Concatenation Fusion Analysis")

print("\nLoading data...")
train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")

tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()
label_encoder = LabelEncoder()
label_encoder.fit(tag_vocab)

train_data["EncodedTags"] = label_encoder.transform(train_data["language"])
test_data["EncodedTags"] = label_encoder.transform(test_data["language"])

num_classes = len(tag_vocab)

print(f"Number of classes: {num_classes}")
print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

# TF-IDF MODEL

print("\n" + "="*80)
print("TRAINING TF-IDF MODEL (Deterministic)")
print("="*80)

best_token_pattern = r'(\b[A-Za-z_]\w*\b|[!\#\$%\&\*\+:\-\./<=>\?@\\\^_\|\~]+|[ \t\(\),;\{\}\[\]`"\'])'

def preprocess(x):
    """Your preprocessing function"""
    return pd.Series(x).replace(r'\b([A-Za-z])\1+\b', '', regex=True)\
        .replace(r'\b[A-Za-z]\b', '', regex=True)

print("Training TF-IDF model...")
transformer = FunctionTransformer(preprocess)
vectorizer = TfidfVectorizer(
    token_pattern=best_token_pattern,
    max_features=10000,
    lowercase=False
)

base_estimator = RandomForestClassifier(
    n_jobs=4,
    random_state=42 
)

tfidf_pipeline = Pipeline([
    ('preprocessing', transformer),
    ('vectorizer', vectorizer),
    ('clf', OneVsRestClassifier(base_estimator))
])

best_params = {
    'clf__estimator__criterion': 'gini',
    'clf__estimator__max_features': 'log2',
    'clf__estimator__min_samples_split': 3,
    'clf__estimator__n_estimators': 400
}
tfidf_pipeline.set_params(**best_params)

tfidf_pipeline.fit(train_data["code"], train_data["EncodedTags"])


tfidf_predictions = tfidf_pipeline.predict(test_data["code"])
tfidf_probabilities = tfidf_pipeline.predict_proba(test_data["code"])
tfidf_accuracy = accuracy_score(test_data["EncodedTags"], tfidf_predictions)

print(f"TF-IDF Model Accuracy: {tfidf_accuracy:.4f}")

class weightedaverage:
    """weighted average fusion with alpha optimization"""
    
    @staticmethod
    def weighted_average_fusion(tfidf_probs, codebert_probs, alpha):
        """
        Simple weighted fusion: alpha * TF-IDF + (1-alpha) * CodeBERT
        """
        fused_probs = alpha * tfidf_probs + (1 - alpha) * codebert_probs
        predictions = np.argmax(fused_probs, axis=1)
        return predictions, fused_probs
    
    @staticmethod
    def find_optimal_alpha_grid(tfidf_probs, codebert_probs, true_labels, 
                               search_range=np.arange(0, 1.01, 0.01)):
        """Find optimal alpha using grid search with detailed analysis"""
        best_accuracy = 0
        best_alpha = 0.5
        alpha_results = []
        
        for alpha in tqdm(search_range, desc="Grid search for optimal alpha", leave=False):
            predictions, _ = weightedaverage.weighted_average_fusion(
                tfidf_probs, codebert_probs, alpha
            )
            accuracy = accuracy_score(true_labels, predictions)
            alpha_results.append((alpha, accuracy))
            
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_alpha = alpha
        
        return best_alpha, best_accuracy, alpha_results
    
    @staticmethod
    def analyze_fusion_curve(tfidf_probs, codebert_probs, true_labels, 
                            search_points=21):
        """Analyze the fusion accuracy curve vs alpha"""
        alphas = np.linspace(0, 1, search_points)
        accuracies = []
        
        for alpha in alphas:
            predictions, _ = weightedaverage.weighted_average_fusion(
                tfidf_probs, codebert_probs, alpha
            )
            accuracies.append(accuracy_score(true_labels, predictions))
        
        return alphas, np.array(accuracies)
    
    @staticmethod
    def get_fusion_improvement(tfidf_probs, codebert_probs, true_labels):
        """Calculate improvement metrics"""
        # Individual model accuracies
        tfidf_preds = np.argmax(tfidf_probs, axis=1)
        codebert_preds = np.argmax(codebert_probs, axis=1)
        
        tfidf_acc = accuracy_score(true_labels, tfidf_preds)
        codebert_acc = accuracy_score(true_labels, codebert_preds)
        
        # Find best fusion
        best_alpha, best_fusion_acc, _ = weightedaverage.find_optimal_alpha_grid(
            tfidf_probs, codebert_probs, true_labels, 
            search_range=np.arange(0, 1.01, 0.02) 
        )
        
        best_single_acc = max(tfidf_acc, codebert_acc)
        improvement = best_fusion_acc - best_single_acc
        
        return {
            'tfidf_accuracy': tfidf_acc,
            'codebert_accuracy': codebert_acc,
            'best_single_accuracy': best_single_acc,
            'optimal_alpha': best_alpha,
            'fusion_accuracy': best_fusion_acc,
            'absolute_improvement': improvement,
            'relative_improvement': improvement / (1 - best_single_acc) * 100
        }

# CODEBERT MODEL

class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = self.tokenizer(
            row["code"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedTags"], dtype=torch.long)
        }

class CodeBERTStage1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
        self.classifier = nn.Linear(
            self.codebert.config.hidden_size,
            num_classes
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        token_embeddings = outputs.last_hidden_state
        attention_mask = attention_mask.unsqueeze(-1)
        pooled_output = (token_embeddings * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
        logits = self.classifier(pooled_output)
        return logits

class CodeBERT_RI_Transformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.codebert = AutoModel.from_pretrained(
            "microsoft/codebert-base",
            output_hidden_states=True
        )
        hidden = self.codebert.config.hidden_size
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            batch_first=True
        )
        self.layer_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        
        self.att_fc = nn.Linear(hidden, hidden)
        self.context_vector = nn.Parameter(torch.randn(hidden))
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states = outputs.hidden_states[1:]
        attention_mask_exp = attention_mask.unsqueeze(-1)
        
        layerwise_embeddings = []
        for layer in hidden_states:
            pooled = (layer * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
            layerwise_embeddings.append(pooled)
        
        layer_sequence = torch.stack(layerwise_embeddings, dim=1)
        h = self.layer_transformer(layer_sequence)
        
        u = torch.tanh(self.att_fc(h))
        scores = torch.matmul(u, self.context_vector)
        alpha = torch.softmax(scores, dim=1)
        x_out = torch.sum(h * alpha.unsqueeze(-1), dim=1)
        
        return self.classifier(x_out)

# MULTI-SEED FUSION EXPERIMENT

print("\n" + "="*80)
print("MULTI-SEED CONCATENATION FUSION EXPERIMENT")
print("="*80)

results_summary = []
detailed_alpha_curves = []

for seed_idx, seed in enumerate(SEEDS[:NUM_SEEDS]):
    print(f"\n{'='*60}")
    print(f"SEED {seed_idx+1}/{NUM_SEEDS}: {seed}")
    print(f"{'='*60}")
    
    set_seed(seed)
    
    print("Training CodeBERT...")
    
    tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
    
    train_dataset = CodeDataset(train_data, tokenizer)
    test_dataset = CodeDataset(test_data, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    # Stage 1 Training
    stage1_model = CodeBERTStage1(num_classes).to(DEVICE)
    optimizer = optim.AdamW(stage1_model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    
    stage1_model.train()
    epochs_stage1 = 3
    
    for epoch in range(epochs_stage1):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 Epoch {epoch+1}", leave=False):
            optimizer.zero_grad()
            logits = stage1_model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            loss = criterion(logits, batch["labels"].to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    # Save Stage 1 weights
    stage1_path = f"/home/aman_swaraj/Downloads/Codelite/codebert_stage1_seed{seed}_concat.pt"
    torch.save(stage1_model.codebert.state_dict(), stage1_path)
    
    # Stage 2 Training
    stage2_model = CodeBERT_RI_Transformer(num_classes).to(DEVICE)
    stage2_model.codebert.load_state_dict(torch.load(stage1_path))
    
    for param in stage2_model.codebert.parameters():
        param.requires_grad = False
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, stage2_model.parameters()),
        lr=1e-4
    )
    
    epochs_stage2 = 5
    stage2_model.train()
    
    for epoch in range(epochs_stage2):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 Epoch {epoch+1}", leave=False):
            optimizer.zero_grad()
            logits = stage2_model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            loss = criterion(logits, batch["labels"].to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    # EVALUATE CODEBERT
    
    stage2_model.eval()
    all_codebert_preds = []
    all_codebert_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            logits = stage2_model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            all_codebert_preds.extend(preds.cpu().numpy())
            all_codebert_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    
    codebert_accuracy = accuracy_score(all_labels, all_codebert_preds)
    codebert_probs = np.array(all_codebert_probs)
    
    print(f"CodeBERT Accuracy (seed {seed}): {codebert_accuracy:.4f}")
    
    # FUSION ANALYSIS
    
    
    print("Analyzing concatenation fusion...")
    
    fusion_analysis = weightedaverage.get_fusion_improvement(
        tfidf_probabilities, codebert_probs, test_data["EncodedTags"]
    )
    
    #fusion curve for visualization
    alphas, accuracies = weightedaverage.analyze_fusion_curve(
        tfidf_probabilities, codebert_probs, test_data["EncodedTags"], 
        search_points=21
    )
    
    detailed_alpha_curves.append({
        'seed': seed,
        'alphas': alphas,
        'accuracies': accuracies
    })
    
    # results
    seed_results = {
        'seed': seed,
        'tfidf_accuracy': tfidf_accuracy,
        'codebert_accuracy': codebert_accuracy,
        'best_single_accuracy': fusion_analysis['best_single_accuracy'],
        'optimal_alpha': fusion_analysis['optimal_alpha'],
        'fusion_accuracy': fusion_analysis['fusion_accuracy'],
        'absolute_improvement': fusion_analysis['absolute_improvement'],
        'relative_improvement': fusion_analysis['relative_improvement'],
        'model_agreement': np.mean(tfidf_predictions == np.array(all_codebert_preds)),
        'fusion_beats_both': fusion_analysis['fusion_accuracy'] > fusion_analysis['best_single_accuracy']
    }
    
    results_summary.append(seed_results)
    
    print(f"  TF-IDF: {tfidf_accuracy:.4f}")
    print(f"  CodeBERT: {codebert_accuracy:.4f}")
    print(f"  Best Single: {fusion_analysis['best_single_accuracy']:.4f}")
    print(f"  Optimal alpha: {fusion_analysis['optimal_alpha']:.3f}")
    print(f"  Concatenation Fusion: {fusion_analysis['fusion_accuracy']:.4f}")
    print(f"  Absolute Improvement: {fusion_analysis['absolute_improvement']:.4f}")
    print(f"  Relative Improvement: {fusion_analysis['relative_improvement']:.1f}%")
    print(f"  Fusion beats best single: {'YES' if seed_results['fusion_beats_both'] else 'NO'}")

# RESULTS ANALYSIS

print("\n" + "="*80)
print("COMPREHENSIVE CONCATENATION FUSION ANALYSIS")
print("="*80)

results_df = pd.DataFrame(results_summary)

print("\nAverage Performance Across Seeds:")
print("-" * 60)

summary_stats = {
    'Metric': [
        'TF-IDF Accuracy',
        'CodeBERT Accuracy (Avg)',
        'Best Single Model (Avg)',
        'Concatenation Fusion (Avg)',
        'Optimal Alpha (Avg)',
        'Absolute Improvement (Avg)',
        'Relative Improvement (Avg)',
        'Model Agreement (Avg)',
        'Fusion Success Rate'
    ],
    'Mean': [
        results_df['tfidf_accuracy'].mean(),
        results_df['codebert_accuracy'].mean(),
        results_df['best_single_accuracy'].mean(),
        results_df['fusion_accuracy'].mean(),
        results_df['optimal_alpha'].mean(),
        results_df['absolute_improvement'].mean(),
        results_df['relative_improvement'].mean(),
        results_df['model_agreement'].mean(),
        results_df['fusion_beats_both'].mean() * 100
    ],
    'Std': [
        results_df['tfidf_accuracy'].std(),
        results_df['codebert_accuracy'].std(),
        results_df['best_single_accuracy'].std(),
        results_df['fusion_accuracy'].std(),
        results_df['optimal_alpha'].std(),
        results_df['absolute_improvement'].std(),
        results_df['relative_improvement'].std(),
        results_df['model_agreement'].std(),
        0
    ],
    'Min': [
        results_df['tfidf_accuracy'].min(),
        results_df['codebert_accuracy'].min(),
        results_df['best_single_accuracy'].min(),
        results_df['fusion_accuracy'].min(),
        results_df['optimal_alpha'].min(),
        results_df['absolute_improvement'].min(),
        results_df['relative_improvement'].min(),
        results_df['model_agreement'].min(),
        results_df['fusion_beats_both'].sum()
    ],
    'Max': [
        results_df['tfidf_accuracy'].max(),
        results_df['codebert_accuracy'].max(),
        results_df['best_single_accuracy'].max(),
        results_df['fusion_accuracy'].max(),
        results_df['optimal_alpha'].max(),
        results_df['absolute_improvement'].max(),
        results_df['relative_improvement'].max(),
        results_df['model_agreement'].max(),
        results_df['fusion_beats_both'].sum()
    ]
}

summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))

# Statistical significance test
print("\n" + "-" * 60)
print("STATISTICAL SIGNIFICANCE TEST")
print("-" * 60)

from scipy import stats

improvements = results_df['absolute_improvement']
t_stat, p_value = stats.ttest_1samp(improvements, 0)

print(f"Mean absolute improvement: {improvements.mean():.4f} ± {improvements.std():.4f}")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.6f}")

if p_value < 0.05:
    significance = "statistically significant"
    sig_symbol = "✓"
else:
    significance = "not statistically significant"
    sig_symbol = "✗"

print(f"{sig_symbol} The improvement is {significance} (p < 0.05)")

success_rate = results_df['fusion_beats_both'].mean() * 100
print(f"\nFusion Success Rate: {success_rate:.1f}% ({results_df['fusion_beats_both'].sum()}/{NUM_SEEDS} seeds)")

print(f"\nOptimal Alpha Analysis:")
print(f"  Mean: {results_df['optimal_alpha'].mean():.3f}")
print(f"  Std: {results_df['optimal_alpha'].std():.3f}")
print(f"  Range: [{results_df['optimal_alpha'].min():.3f}, {results_df['optimal_alpha'].max():.3f}]")
print(f"  Interpretation: TF-IDF weight {results_df['optimal_alpha'].mean()*100:.1f}%, CodeBERT weight {(1-results_df['optimal_alpha'].mean())*100:.1f}%")

# VISUALIZATION

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print("\nGenerating concatenation fusion visualizations...")
    
    plt.figure(figsize=(14, 6))
    
    seeds_list = [f"Seed {s}" for s in SEEDS[:NUM_SEEDS]]
    x = np.arange(NUM_SEEDS)
    width = 0.2
    
    plt.bar(x - width*1.5, results_df['tfidf_accuracy'], width, 
            label='TF-IDF', color='#FF6B6B', alpha=0.8)
    plt.bar(x - width*0.5, results_df['codebert_accuracy'], width, 
            label='CodeBERT', color='#4ECDC4', alpha=0.8)
    plt.bar(x + width*0.5, results_df['best_single_accuracy'], width, 
            label='Best Single', color='#FFD166', alpha=0.8)
    plt.bar(x + width*1.5, results_df['fusion_accuracy'], width, 
            label='Concatenation Fusion', color='#06D6A0', alpha=0.8)
    
    plt.xlabel('Random Seed')
    plt.ylabel('Accuracy')
    plt.title(f'Concatenation Fusion Performance Across {NUM_SEEDS} Seeds')
    plt.xticks(x, seeds_list)
    plt.legend()
    plt.grid(True, alpha=0.3, linestyle='--')
    

    for i in range(NUM_SEEDS):
        improvement = results_df['absolute_improvement'].iloc[i]
        if improvement > 0:
            plt.text(i + width*1.5, results_df['fusion_accuracy'].iloc[i] + 0.002,
                    f"+{improvement:.4f}", ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig('concatenation_fusion_performance.png', dpi=300, bbox_inches='tight')
    print("  Saved: concatenation_fusion_performance.png")
    
    plt.figure(figsize=(12, 8))
    
    for idx, curve_data in enumerate(detailed_alpha_curves):
        plt.plot(curve_data['alphas'], curve_data['accuracies'], 
                label=f'Seed {SEEDS[idx]}', linewidth=2, alpha=0.7)
        
        optimal_idx = np.argmax(curve_data['accuracies'])
        plt.scatter(curve_data['alphas'][optimal_idx], 
                   curve_data['accuracies'][optimal_idx], 
                   s=100, zorder=5)
    
    plt.axvline(x=results_df['optimal_alpha'].mean(), color='red', 
                linestyle='--', linewidth=2, label=f'Mean optimal α = {results_df["optimal_alpha"].mean():.3f}')
    
    plt.xlabel('Alpha (TF-IDF weight)')
    plt.ylabel('Accuracy')
    plt.title('Concatenation Fusion Accuracy vs Alpha (TF-IDF weight)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    textstr = '\n'.join((
        f'Mean optimal α: {results_df["optimal_alpha"].mean():.3f}',
        f'Std optimal α: {results_df["optimal_alpha"].std():.3f}',
        f'Mean improvement: {results_df["absolute_improvement"].mean():.4f}',
        f'Success rate: {success_rate:.1f}%'
    ))
    
    plt.gca().text(0.05, 0.95, textstr, transform=plt.gca().transAxes,
                   fontsize=10, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig('concatenation_fusion_curves.png', dpi=300, bbox_inches='tight')
    print("  Saved: concatenation_fusion_curves.png")
    
    
    plt.figure(figsize=(10, 6))
    
    plt.subplot(1, 2, 1)
    improvements_pct = results_df['relative_improvement']
    colors = ['#06D6A0' if imp > 0 else '#EF476F' for imp in improvements_pct]
    bars = plt.bar(range(1, NUM_SEEDS + 1), improvements_pct, color=colors)
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    plt.xlabel('Seed')
    plt.ylabel('Relative Improvement (%)')
    plt.title('Fusion Improvement by Seed')
    plt.grid(True, alpha=0.3, linestyle='--')
    
    
    for i, (bar, imp) in enumerate(zip(bars, improvements_pct)):
        plt.text(bar.get_x() + bar.get_width()/2, 
                imp + (0.5 if imp >= 0 else -1.5),
                f'{imp:.1f}%', ha='center', va='bottom' if imp >= 0 else 'top')
    
    plt.subplot(1, 2, 2)
    plt.hist(results_df['optimal_alpha'], bins=10, alpha=0.7, 
             color='#118AB2', edgecolor='black')
    plt.axvline(x=results_df['optimal_alpha'].mean(), color='red', 
                linestyle='--', linewidth=2, label=f'Mean: {results_df["optimal_alpha"].mean():.3f}')
    plt.xlabel('Optimal Alpha (TF-IDF weight)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Optimal Alpha Values')
    plt.legend()
    plt.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig('concatenation_fusion_improvements.png', dpi=300, bbox_inches='tight')
    print("  Saved: concatenation_fusion_improvements.png")
    
    plt.show()
    
except ImportError:
    print("\nVisualization libraries not available. Skipping plots.")

print("\n" + "="*80)
print("FUSION EXPERIMENT COMPLETE!")
print("="*80)